In [32]:
import gzip
import json
import re
import pandas as pd
from lxml import etree
from pathlib import Path
from difflib import SequenceMatcher
from collections import Counter
import os
import sys

def normalize_for_matching(text):
    """Normalize text for comparison: lowercase, remove non-alphanumeric, collapse whitespace."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def build_word_frequency_corpus(corpus_dir: Path) -> Counter:
    """
    Builds a word frequency counter from a directory of text files,
    including gzipped JSONs (like VILA) and plain text.
    Crucially, applies basic ligature fixes to corpus content before counting words
    to ensure full words like "unfortunately" are counted even if they come from
    hyphenated/ligated forms in the raw corpus source.
    """
    word_counts = Counter()
    if not corpus_dir.exists():
        print(f"Warning: Corpus path '{corpus_dir}' does not exist. Cannot build frequency corpus.", file=sys.stderr)
        return word_counts

    print(f"Building word frequency corpus from '{corpus_dir}'...")
    file_count = 0
    for root, _, files in os.walk(corpus_dir):
        for file_name in files:
            file_path = Path(root) / file_name
            content = ""
            try:
                if file_path.suffix == '.gz':
                    if 'json' in file_path.name.lower():
                         with gzip.open(file_path, 'rt', encoding='utf-8') as f:
                             data = json.load(f)
                             content = data.get("symbols", "")
                    else:
                        with gzip.open(file_path, 'rt', encoding='utf-8') as f:
                            content = f.read()
                elif file_path.suffix == '.json':
                    with open(file_path, 'r', encoding='utf-8') as f:
                        data = json.load(f)
                        content = data.get("symbols", "")
                else:
                    with open(file_path, 'r', encoding='utf-8') as f:
                        content = f.read()

                # --- NEW: Apply basic text cleaning to corpus content BEFORE counting ---
                # This ensures words like "unfortunately" get into the corpus
                # even if they were originally "Unfor-\ntunately" or had ligatures.
                temp_cleaned_content = content.replace('ï¬', 'ffi')
                temp_cleaned_content = temp_cleaned_content.replace('ï¬‚', 'ffl')
                temp_cleaned_content = temp_cleaned_content.replace('â€¢', '•')

                # A more aggressive dehyphenation for corpus building (might not be needed if this fix works)
                # This is a simpler dehyphenation just for populating the corpus, not the main dehyphenator.
                temp_cleaned_content = re.sub(r'([a-zA-Z]+)-\s*\n\s*([a-zA-Z]+)', r'\1\2', temp_cleaned_content)


                words = re.findall(r'\b[a-z]+\b', temp_cleaned_content.lower()) # Count lowercased words
                word_counts.update(words)
                file_count += 1
            except Exception as e:
                print(f"Error reading corpus file '{file_path}' for frequency building: {e}", file=sys.stderr)
    print(f"Finished building corpus from {file_count} files. Loaded {len(word_counts)} unique words.")
    return word_counts

def dehyphenate_text_with_corpus(text: str, word_frequencies: Counter) -> str:
    """
    Dehyphenates text by checking word frequencies for hyphenated words at line breaks.
    Attempts to preserve original casing, especially for sentence-starting words.
    Includes a fallback for zero frequencies if the word looks like a clear dehyphenation candidate.
    """
    if not isinstance(text, str):
        return ""

    if not word_frequencies:
        print("Warning: Word frequency corpus not loaded. Dehyphenation will rely on basic heuristics.", file=sys.stderr)
        # If no corpus, we'll try a simpler dehyphenation
        return re.sub(r'([a-zA-Z]+)-\s*\n\s*([a-zA-Z]+)', r'\1\2', text)


    lines = text.split('\n')
    processed_lines = []
    i = 0
    while i < len(lines):
        current_line = lines[i]

        if i + 1 < len(lines):
            next_line = lines[i+1]

            match_candidates = list(re.finditer(r'([a-zA-Z]+)-(\s*)$', current_line))

            if match_candidates:
                match = match_candidates[-1]
                original_first_part_full = match.group(1)
                trailing_spaces = match.group(2)

                next_word_match = re.match(r'^\s*([a-zA-Z]+)\b', next_line)

                if next_word_match:
                    original_second_part_full = next_word_match.group(1)

                    # Removed: print(f"DEBUG: Found potential hyphenation: '{original_first_part_full}-' and '{original_second_part_full}'")

                    candidate_dehyphenated_lower = (original_first_part_full + original_second_part_full).lower()
                    candidate_hyphenated_retained_lower = (original_first_part_full + '-' + original_second_part_full).lower()

                    freq_dehyphenated = word_frequencies.get(candidate_dehyphenated_lower, 0)
                    freq_hyphenated_retained = word_frequencies.get(candidate_hyphenated_retained_lower, 0)

                    # Removed: print(f"DEBUG: Frequencies - Dehyphenated ('{candidate_dehyphenated_lower}'): {freq_dehyphenated}, Hyphenated Retained ('{candidate_hyphenated_retained_lower}'): {freq_hyphenated_retained}")

                    should_dehyphenate = False
                    if (freq_dehyphenated > 0 and freq_hyphenated_retained == 0) or \
                       (freq_dehyphenated > freq_hyphenated_retained * 2 and freq_dehyphenated > 0):
                        should_dehyphenate = True
                        # Removed: print("DEBUG: Decision: Dehyphenate based on frequency comparison.")
                    elif freq_dehyphenated == 0 and freq_hyphenated_retained == 0:
                        should_dehyphenate = True
                        # Removed: print("DEBUG: Decision: Frequencies are both 0. Attempting heuristic dehyphenation.")

                    if should_dehyphenate:
                        first_char_capitalized = original_first_part_full[0].isupper()
                        combined_word = original_first_part_full + original_second_part_full

                        if first_char_capitalized:
                            resolved_word = combined_word[0].upper() + combined_word[1:]
                            # Removed: print(f"DEBUG: First part capitalized. Resolved to: '{resolved_word}' (Ensured first char upper)")
                        else:
                            resolved_word = combined_word
                            # Removed: print(f"DEBUG: First part lowercase. Resolved to: '{resolved_word}' (Using exact combined casing)")

                        current_line_modified = current_line[:match.start()] + resolved_word + trailing_spaces
                        next_line_modified = re.sub(r'^\s*' + re.escape(original_second_part_full) + r'\b', '', next_line, 1)

                        processed_lines.append(current_line_modified)
                        lines[i+1] = next_line_modified
                        i += 1
                        # Removed: print(f"DEBUG: Successfully dehyphenated and advanced line pointer.")
                        continue
                else:
                    pass # Removed: print(f"DEBUG: Next line does not start with an alphabetic word after '{original_first_part_full}-'. No dehyphenation.")
            else:
                pass # No hyphenated word at the end of the current line

        processed_lines.append(current_line)
        i += 1

    return '\n'.join(processed_lines)


def extract_vila_sections(json_content):
    if not isinstance(json_content, str):
        return []
    sections = []
    pattern = re.compile(
        r'^\s*(?P<number>\d+(?:\.\d+)*\s*)?'
        r'(?P<heading_text>[A-Z][A-Z\s]+(?:[A-Z]|\d)*)'
        r'\s*\n+'
        r'(?P<content>.*?)'
        r'(?=\n^\s*(?:\d+(?:\.\d+)*\s*)?[A-Z][A-Z\s]+(?:[A-Z]|\d)*\s*\n+|\Z)',
        re.MULTILINE | re.DOTALL
    )
    for match in pattern.finditer(json_content):
        heading = match.group('heading_text').strip()
        text = match.group('content').strip()
        num_part = match.group('number')
        level = num_part.count('.') + 1 if num_part else 0
        sections.append((level, heading, text))
    return sections


def extract_grobid_headings(file_path):
    if not isinstance(file_path, Path):
        file_path = Path(file_path)
    try:
        try:
            with open(file_path, 'rb') as f:
                tree = etree.parse(f)
        except etree.XMLSyntaxError:
            with gzip.open(file_path, 'rb') as f:
                tree = etree.parse(f)
        ns = {'tei': 'http://www.tei-c.org/ns/1.0'}
        headings = []
        for head_element in tree.xpath('//tei:body//tei:head[@n]', namespaces=ns):
            n_value = head_element.get('n')
            head_text_elements = head_element.xpath('./text()', namespaces=ns)
            head_text = head_text_elements[0].strip() if head_text_elements else ''
            if not head_text:
                continue
            parts = n_value.split('.')
            level = len(parts)
            if level == 1:
                headings.append({
                    'level': level,
                    'n_value': n_value,
                    'text': head_text,
                    'subheadings': []
                })
            else:
                parent_n = '.'.join(parts[:-1])
                found_parent = False
                for main_heading in headings:
                    if main_heading['n_value'] == parent_n:
                        main_heading['subheadings'].append({
                            'level': level,
                            'n_value': n_value,
                            'text': head_text
                        })
                        found_parent = True
                        break
                if not found_parent:
                    headings.append({
                        'level': level,
                        'n_value': n_value,
                        'text': head_text,
                        'subheadings': []
                    })
        return headings
    except Exception as e:
        print(f"Error processing GROBID file '{file_path}': {e}", file=sys.stderr)
        return []

def process_files_to_csv(vila_dir, grobid_dir, output_csv, min_score=0.6, corpus_for_dehyphenation=None):
    vila_dir = Path(vila_dir)
    grobid_dir = Path(grobid_dir)
    global_word_frequencies = Counter()
    if corpus_for_dehyphenation:
        global_word_frequencies = build_word_frequency_corpus(Path(corpus_for_dehyphenation))
        print(f"Successfully loaded {len(global_word_frequencies)} unique words for dehyphenation.")
    else:
        print("No corpus path provided for dehyphenation. Dehyphenation will be skipped.", file=sys.stderr)

    vila_files = sorted(list(vila_dir.glob("*.json.gz")))
    grobid_files = sorted(list(grobid_dir.glob("*.tei*")))

    grobid_map = {}
    for g_path in grobid_files:
        base_name = g_path.name.split('.grobid')[0]
        grobid_map[base_name] = g_path

    all_matched_records = []
    processed_paper_ids = set()

    print(f"\nStarting to process {len(vila_files)} VILA files...")
    for v_path in vila_files:
        paper_id = v_path.stem.split('.')[0]

        if paper_id in processed_paper_ids:
            print(f"Skipping already processed paper: {paper_id}", file=sys.stderr)
            continue

        print(f"Processing paper: {paper_id}")

        if paper_id not in grobid_map:
            print(f"Warning: No matching GROBID file found for '{paper_id}'. Skipping.", file=sys.stderr)
            continue

        g_path = grobid_map[paper_id]

        try:
            with gzip.open(v_path, 'rt', encoding='utf-8') as f:
                vila_data = json.load(f)
                vila_raw_text = vila_data.get("symbols", "")

                vila_processed_text = vila_raw_text.replace('ï¬', 'ffi')
                vila_processed_text = vila_processed_text.replace('ï¬‚', 'ffl')
                vila_processed_text = vila_processed_text.replace('â€¢', '•')

                if global_word_frequencies:
                    print(f"--- Dehyphenating for paper {paper_id} ---")
                    vila_processed_text = dehyphenate_text_with_corpus(vila_processed_text, global_word_frequencies)
                    print(f"--- Dehyphenation complete for paper {paper_id} ---")

                vila_sections = extract_vila_sections(vila_processed_text)
                if not vila_sections:
                    print(f"Warning: No VILA sections extracted for {paper_id}. Skipping matching.", file=sys.stderr)
                    continue

            grobid_headings = extract_grobid_headings(g_path)
            if not grobid_headings:
                print(f"Warning: No GROBID headings extracted for {paper_id}. Skipping matching.", file=sys.stderr)
                continue

            for grobid_main_heading in grobid_headings:
                grobid_heading_text = grobid_main_heading['text']
                normalized_grobid_heading = normalize_for_matching(grobid_heading_text)

                best_vila_match = None
                highest_match_score = 0

                for vila_level, vila_heading_text, vila_section_content in vila_sections:
                    normalized_vila_heading = normalize_for_matching(vila_heading_text)

                    match_score = SequenceMatcher(
                        None,
                        normalized_vila_heading,
                        normalized_grobid_heading
                    ).ratio()

                    if match_score > highest_match_score and match_score >= min_score:
                        highest_match_score = match_score
                        best_vila_match = (vila_heading_text, vila_section_content)

                subheadings_str = "; ".join(
                    f"{sub['n_value']}: {sub['text']}"
                    for sub in grobid_main_heading['subheadings']
                )

                if best_vila_match:
                    all_matched_records.append({
                        "paper_id": paper_id,
                        "section_name_grobid": grobid_heading_text,
                        "section_content_vila": best_vila_match[1],
                        "subheadings_grobid": subheadings_str
                    })

            processed_paper_ids.add(paper_id)

        except Exception as e:
            print(f"Critical error processing paper '{paper_id}': {e}", file=sys.stderr)

    if all_matched_records:
        df = pd.DataFrame(all_matched_records)
        df.to_csv(output_csv, index=False, encoding='utf-8')
        print(f"\nProcessing complete! Results saved to '{output_csv}'")
        print(f"Total unique papers with matched sections: {len(processed_paper_ids)}")
        print(f"Total sections matched and exported: {len(df)}")
    else:
        print("\nNo matching data found to export after processing all files.", file=sys.stderr)

if __name__ == "__main__":
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print("Google Drive mounted successfully.")
    except ImportError:
        print("Not running in Google Colab environment. Skipping Google Drive mount.")
    except Exception as e:
        print(f"Error mounting Google Drive: {e}", file=sys.stderr)

    VILA_INPUT_DIR = "/content/drive/MyDrive/vila_test"
    GROBID_INPUT_DIR = "/content/drive/MyDrive/grobid_test"
    OUTPUT_CSV_FILE = "matched_sections_cleaned_final.csv"
    MATCHING_MIN_SCORE = 0.6
    CORPUS_FOR_DEHYPHENATION = VILA_INPUT_DIR

    print("\n--- Starting Text Processing Script ---")
    process_files_to_csv(
        vila_dir=VILA_INPUT_DIR,
        grobid_dir=GROBID_INPUT_DIR,
        output_csv=OUTPUT_CSV_FILE,
        min_score=MATCHING_MIN_SCORE,
        corpus_for_dehyphenation=CORPUS_FOR_DEHYPHENATION
    )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted successfully.

--- Starting Text Processing Script ---
Building word frequency corpus from '/content/drive/MyDrive/vila_test'...
Finished building corpus from 2 files. Loaded 2579 unique words.
Successfully loaded 2579 unique words for dehyphenation.

Starting to process 2 VILA files...
Processing paper: _0kaDkv3dVf
--- Dehyphenating for paper _0kaDkv3dVf ---
--- Dehyphenation complete for paper _0kaDkv3dVf ---
Processing paper: ztMLindFLWR
--- Dehyphenating for paper ztMLindFLWR ---
--- Dehyphenation complete for paper ztMLindFLWR ---

Processing complete! Results saved to 'matched_sections_cleaned_final.csv'
Total unique papers with matched sections: 2
Total sections matched and exported: 7

--- Script Finished ---
